In [5]:
# Fix for RTX 50-series (Blackwell / sm_120)
%pip install torch torchvision torchaudio --force-reinstall --index-url https://download.pytorch.org/whl/cu128

^C
Note: you may need to restart the kernel to use updated packages.


# SER — Environment Setup
## AudioGuard FYP | Speech Emotion Recognition Track
**Purpose**: Run this notebook FIRST before any S1–S7 SER notebooks. Installs all dependencies, verifies GPU, downloads TESS and RAVDESS via Kaggle API, checks IEMOCAP path from `.env`, and runs the unified label mapping to print class distribution.

**Run order**: `00_environment_setup → S1 → S2 → S3 → S4 → S5 → S6 → S7 → S_evaluate_all`

In [ ]:
# Cell 1 — Install all SER dependencies
%pip install transformers>=4.40.0 datasets>=2.18.0 accelerate>=0.29.0 \
             librosa>=0.10.0 soundfile>=0.12.0 \
             scikit-learn pandas numpy tqdm kaggle \
             python-dotenv torch torchvision torchaudio \
             matplotlib seaborn jupyter notebook ipywidgets --quiet
%pip install tensorflow keras --quiet
print('✓ All SER dependencies installed')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ All SER dependencies installed


In [1]:
# Cell 2 — Verify PyTorch + CUDA
import torch, sys, psutil, shutil, platform
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
    if vram < 12:
        print('⚠ <12GB VRAM — S2 (Whisper) and S5 (WavLM) will need batch_size=4')
else:
    print('No GPU — S2 (Whisper-Large), S5 (WavLM) are NOT recommended on CPU')

PyTorch version: 2.5.1+cu121
CUDA available:  True
GPU:  NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB
⚠ <12GB VRAM — S2 (Whisper) and S5 (WavLM) will need batch_size=4


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [2]:
# Cell 3 — Verify Kaggle API connection (required for TESS and RAVDESS)
import os, subprocess
kaggle_path = os.path.join(os.path.expanduser('~'), '.kaggle', 'kaggle.json')
if not os.path.exists(kaggle_path):
    raise FileNotFoundError(
        f'kaggle.json NOT found at {kaggle_path}\n'
        'SER notebooks require Kaggle API to download TESS and RAVDESS.\n'
        'Download from: https://www.kaggle.com/settings → API → Create New Token'
    )
print(f'✓ kaggle.json found at {kaggle_path}')
result = subprocess.run(['kaggle', 'datasets', 'list', '--max-size', '1'], capture_output=True, text=True)
if result.returncode == 0:
    print('✓ Kaggle API connection successful')
else:
    print('✗ Kaggle API error:', result.stderr[:300])
    print('Fix: pip install kaggle, ensure kaggle.json path permissions are correct (chmod 600 on Linux/Mac)')

✓ kaggle.json found at C:\Users\User\.kaggle\kaggle.json
✓ Kaggle API connection successful


In [4]:
# Cell 4 — Download TESS (skip if already present)
TESS_DIR = os.path.join('..', '..', 'datasets', 'tess')
os.makedirs(TESS_DIR, exist_ok=True)

tess_files = []
for root, dirs, files in os.walk(TESS_DIR):
    tess_files.extend([f for f in files if f.endswith('.wav')])

if len(tess_files) > 100:
    print(f'✓ TESS already downloaded ({len(tess_files)} .wav files) — skipping download')
else:
    print(f'Downloading TESS from Kaggle (~50MB)...')
    try:
        os.system(f'kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -p {TESS_DIR} --unzip --quiet')
        tess_files = []
        for root, dirs, files in os.walk(TESS_DIR):
            tess_files.extend([f for f in files if f.endswith('.wav')])
        print(f'✓ TESS downloaded: {len(tess_files)} .wav files')
    except Exception as e:
        print(f'✗ TESS download failed: {e}')
        print('Manual download: kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess')

✓ TESS downloaded: 5600 .wav files


In [5]:
# Cell 5 — Download RAVDESS (skip if already present)
RAVDESS_DIR = os.path.join('..', '..', 'datasets', 'ravdess')
os.makedirs(RAVDESS_DIR, exist_ok=True)

ravdess_files = []
for root, dirs, files in os.walk(RAVDESS_DIR):
    ravdess_files.extend([f for f in files if f.endswith('.wav')])

if len(ravdess_files) > 100:
    print(f'✓ RAVDESS already downloaded ({len(ravdess_files)} .wav files) — skipping')
else:
    print('Downloading RAVDESS from Kaggle (~500MB)...')
    try:
        os.system(f'kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -p {RAVDESS_DIR} --unzip --quiet')
        ravdess_files = []
        for root, dirs, files in os.walk(RAVDESS_DIR):
            ravdess_files.extend([f for f in files if f.endswith('.wav')])
        print(f'✓ RAVDESS downloaded: {len(ravdess_files)} .wav files')
    except Exception as e:
        print(f'✗ RAVDESS download failed: {e}')

✓ RAVDESS downloaded: 2880 .wav files


In [1]:
# Cell 6 — Check IEMOCAP path (optional — warn but never error)
import logging
try:
    from dotenv import load_dotenv
    env_path = os.path.join('..', '..', '.env')
    if os.path.exists(env_path):
        load_dotenv(env_path)
    iemocap_path = os.environ.get('IEMOCAP_PATH', '')
    if iemocap_path and os.path.exists(iemocap_path):
        print(f'✓ IEMOCAP found at: {iemocap_path}')
    elif iemocap_path:
        logging.warning(f'IEMOCAP_PATH set but path does not exist: {iemocap_path}')
        print(f'⚠ IEMOCAP_PATH set but not found: {iemocap_path}')
    else:
        logging.warning('IEMOCAP_PATH not set — IEMOCAP will be skipped in all SER notebooks')
        print('⚠ IEMOCAP_PATH not set in .env — IEMOCAP will be skipped (not required)')
        print('  To enable: set IEMOCAP_PATH=<path_to_IEMOCAP> in AudioGuardMP_2026/.env')
except Exception as e:
    logging.warning(f'Could not check IEMOCAP path: {e}')
    print(f'⚠ Could not load .env: {e} — IEMOCAP will be skipped')

⚠ Could not load .env: name 'os' is not defined — IEMOCAP will be skipped


In [2]:
# Cell 7 — Unified label mapping for all downloaded audio files
import os

EMOTION_MAP = {
    # RAVDESS numeric codes (from filename 3rd segment)
    '01': 0,  # neutral
    '02': 0,  # calm → neutral
    '03': 1,  # happy
    '04': 2,  # sad
    '05': 3,  # angry
    '06': 4,  # fearful → fear
    '07': 5,  # disgust
    '08': 6,  # surprised → surprise
    # TESS folder names (from parent directory)
    'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3,
    'fear': 4, 'disgust': 5, 'ps': 6,  # ps = pleasant surprise
}
LABEL_NAMES = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']
print('Unified 7-class emotion mapping:')
for k, v in EMOTION_MAP.items():
    print(f'  {k:12s} → {LABEL_NAMES[v]}')

# Count files per class across both datasets
from collections import Counter
label_counts = Counter()

for root, dirs, files in os.walk(os.path.join('..', '..', 'datasets', 'ravdess')):
    for f in files:
        if f.endswith('.wav'):
            parts = f.replace('.wav', '').split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                if emotion_code in EMOTION_MAP:
                    label_counts[LABEL_NAMES[EMOTION_MAP[emotion_code]]] += 1

for root, dirs, files in os.walk(os.path.join('..', '..', 'datasets', 'tess')):
    for f in files:
        if f.endswith('.wav'):
            folder = os.path.basename(root).lower()
            for key in EMOTION_MAP:
                if folder.endswith(key):
                    label_counts[LABEL_NAMES[EMOTION_MAP[key]]] += 1
                    break

print(f'\nClass distribution across TESS + RAVDESS:')
for label in LABEL_NAMES:
    print(f'  {label:10s}: {label_counts.get(label, 0)}')

Unified 7-class emotion mapping:
  01           → neutral
  02           → neutral
  03           → happy
  04           → sad
  05           → angry
  06           → fear
  07           → disgust
  08           → surprise
  neutral      → neutral
  happy        → happy
  sad          → sad
  angry        → angry
  fear         → fear
  disgust      → disgust
  ps           → surprise

Class distribution across TESS + RAVDESS:
  neutral   : 1376
  happy     : 1184
  sad       : 1184
  angry     : 1184
  fear      : 1184
  disgust   : 1184
  surprise  : 384


In [4]:
# Cell 8 — System summary
import sys, psutil, shutil, platform
print('=== SYSTEM SUMMARY ===')
print(f'OS:         {platform.system()} {platform.version()}')
print(f'Python:     {sys.version}')
print(f'PyTorch:    {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:        {torch.cuda.get_device_name(0)}')
    print(f'VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'RAM:        {psutil.virtual_memory().total / 1e9:.1f} GB total')
print(f'Disk free:  {shutil.disk_usage(".").free / 1e9:.1f} GB')
try:
    import tensorflow as tf
    print(f'TensorFlow: {tf.__version__}')
except ImportError:
    print('TensorFlow: NOT installed (needed for S1 LSTM)')
print()
print('✓ SER environment setup complete — you may now run S1 through S7.')

=== SYSTEM SUMMARY ===
OS:         Windows 10.0.26100
Python:     3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


NameError: name 'torch' is not defined